In [ ]:
import boto3
import pandas as pd
import sqlite3
from io import BytesIO

# Initialize S3 client
s3 = boto3.client('s3')

# S3 bucket and paths
bucket_name = "s3_bucket_name"
input_prefix = "taxi/glue-transformed/"
output_prefix = "taxi/sqlite3db/"
sqlite_db_name = "taxi_data.db"

# Function to list all Parquet files in the specified year and month
def list_parquet_files(bucket, prefix, year, month):
    folder_prefix = f"{prefix}year={year}/month={month:02d}/"
    response = s3.list_objects_v2(Bucket=bucket, Prefix=folder_prefix)
    if 'Contents' in response:
        return [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.parquet')]
    return []

# Function to read a Parquet file from S3 into a Pandas DataFrame
def read_parquet_from_s3(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(response['Body'].read()))

# Function to upload SQLite database to S3
def upload_sqlite_to_s3(bucket, key, db_path):
    with open(db_path, 'rb') as f:
        s3.upload_fileobj(f, bucket, key)


# Create an SQLite database
conn = sqlite3.connect(sqlite_db_name)
cursor = conn.cursor()

# Loop through years and months
for year in [2023, 2024]:
    for month in range(1, 13):
        print(f"Processing year={year}, month={month:02d}...")
        parquet_files = list_parquet_files(bucket_name, input_prefix, year, month)

        for file_key in parquet_files:
            print(f"Reading file: {file_key}")
            # Read Parquet file into a DataFrame
            df = read_parquet_from_s3(bucket_name, file_key)

            # Append data to SQLite database
            df.to_sql('taxi_data', conn, if_exists='append', index=False)

# Create an index on the `pickup_hour` column
print("Creating index on pickup_hour...")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_pickup_hour ON taxi_data (pickup_hour);")

# Create an index on the `pickup_location_id` column
print("Creating index on pickup_location_id...")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_pickup_location_id ON taxi_data (pickup_location_id);")

# Commit and close the SQLite database
conn.commit()
conn.close()

# Upload the SQLite database to S3
output_key = f"{output_prefix}{sqlite_db_name}"
print(f"Uploading SQLite database to S3: {output_key}")
upload_sqlite_to_s3(bucket_name, output_key, sqlite_db_name)
print("Upload complete.")

In [ ]:
import boto3
import sqlite3
import pandas as pd
import tempfile

# S3 bucket and database path
bucket_name = "s3_bucket_name"
sqlite_db_key = "path_to_taxi_data.db" 

# Initialize S3 client
s3 = boto3.client('s3')

# Download the SQLite database from S3 to a temporary file
with tempfile.NamedTemporaryFile(suffix=".db") as temp_file:
    # Download the database file from S3
    s3.download_file(bucket_name, sqlite_db_key, temp_file.name)

    # Connect to the SQLite database
    conn = sqlite3.connect(temp_file.name)

    # Query the database to read the data into a Pandas DataFrame
    query = "SELECT * FROM taxi_data;"  # Replace with your desired query if needed
    df = pd.read_sql_query(query, conn)

    # Close the connection
    conn.close()

# Display the DataFrame
print(df.head())  # Display the first few rows of the DataFrame

In [ ]:
df.shape

In [ ]:
import pandas as pd

In [ ]:
# Model Training with 2023
# Filter the DataFrame for the specified conditions
filtered_df = (
    df.loc[
        (df['pickup_hour'] >= '2023-01-01 00:00:00') &
        (df['pickup_hour'] <= '2024-01-31 23:59:59') &
        (df['pickup_location_id'] in (43, 138, 237)),
        ['pickup_hour', 'pickup_location_id', 'rides']
    ]
    .sort_values(by='pickup_hour')
)

# Display the filtered DataFrame
print(filtered_df)

In [ ]:
# Model Predictions for 2024
# Filter the DataFrame for the specified conditions
filtered_df_2024 = (
    df.loc[
        (df['pickup_hour'] >= '2023-12-04 00:00:00') &
        (df['pickup_hour'] <= '2024-12-31 23:59:59') &
        (df['pickup_location_id'] == 43),
        ['pickup_hour', 'pickup_location_id', 'rides']
    ]
    .sort_values(by='pickup_hour')
)

# Display the filtered DataFrame
print(filtered_df)

In [ ]:
filtered_df_43 = filtered_df[filtered_df["pickup_location_id"]==43]
filtered_df_138 = filtered_df[filtered_df["pickup_location_id"]==138]
filtered_df_237 = filtered_df[filtered_df["pickup_location_id"]==237]

filtered_df_2024_43 = filtered_df_2024[filtered_df_2024["pickup_location_id"]==43]
filtered_df_2024_138 = filtered_df_2024[filtered_df_2024["pickup_location_id"]==138]
filtered_df_2024_237 = filtered_df_2024[filtered_df_2024["pickup_location_id"]==237]




In [ ]:
def create_time_series_features(df, n_lags)   
    """
    Creates features and targets for the provided DataFrame, assuming it contains data for a single location.

    Parameters:
    - df: DataFrame with columns [pickup_hour, rides]
    - n_lags: Number of lag features to create

    Returns:
    - Xy: Features DataFrame with target column
    """
    # Check if enough data
    if len(df) <= n_lags:
        raise ValueError(f"Insufficient data ({len(df)} rows)")

    # Sort the data by pickup_hour
    df = df.sort_values('pickup_hour')
    rides_series = df['rides'].reset_index(drop=True)

    # Create lag features
    lag_data = {}
    for lag in range(1, n_lags + 1):
        lag_data[f'lag_{lag}'] = rides_series.shift(lag)

    # Create features DataFrame
    features_df = pd.DataFrame(lag_data)
    features_df['target'] = rides_series
    features_df['pickup_hour'] = df['pickup_hour'].reset_index(drop=True)

    # Drop rows with NaN values
    features_df = features_df.dropna()

    if len(features_df) == 0:
        raise ValueError("No valid data after dropping NaNs")

    # Create final dataset
    column_order = [f'lag_{i}' for i in range(n_lags, 0, -1)]
    Xy = features_df[column_order].copy()
    Xy['pickup_hour'] = features_df['pickup_hour']
    Xy['target'] = features_df['target'].copy()

    print(f"\nCreated dataset with {len(Xy)} rows")
    print(f"Date range: {Xy['pickup_hour'].min()} to {Xy['pickup_hour'].max()}")

    return Xy

In [ ]:
df_ts_43 = create_time_series_features(filtered_df_43, 24)
df_ts_138 = create_time_series_features(filtered_df_138, 24)
df_ts_237 = create_time_series_features(filtered_df_237, 24)

df_ts_2024_43 = create_time_series_features(filtered_df_2024_43, 24)
df_ts_2024_138 = create_time_series_features(filtered_df_2024_138, 24)
df_ts_2024_237 = create_time_series_features(filtered_df_2024_237, 24)


In [ ]:
# Features and Targets for 2023 modle training
#For location 43
X_43 = df_ts_43.copy()
X_43 = X_43.drop(columns=['pickup_hour', 'target'])
y_43 = df_ts_43['target']

#For location 138
X_138 = df_ts_138.copy()
X_138 = X_138.drop(columns=['pickup_hour', 'target'])
y_138 = df_ts_138['target']

#For location 237
X_237 = df_ts_237.copy()
X_237 = X_237.drop(columns=['pickup_hour', 'target'])
y_237 = df_ts_237['target']


#Features and Targets for 2024 Predictions
#For location 43
X_2024_43 = df_ts_2024_43.copy()
X_2024_43 = X_2024_43.drop(columns=['pickup_hour', 'target'])
y_2024_43 = df_ts_2024_43['target']

#For location 138
X_2024_138 = df_ts_2024_138.copy()
X_2024_138 = X_2024_138.drop(columns=['pickup_hour', 'target'])
y_2024_138 = df_ts_2024_138['target']

#For location 237
X_2024_237 = df_ts_2024_237.copy()
X_2024_237 = X_2024_237.drop(columns=['pickup_hour', 'target'])
y_2024_237 = df_ts_2024_237['target']


## Model 1 LightGBM with 28 day lag features

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import numpy as np

def smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    return np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100

model = lgb.LGBMRegressor()


In [ ]:

#Location 43
model = lgb.LGBMRegressor()
model.fit(X_43, y_43)
y_pred_43 = np.ceil(model.predict(X_43)).astype(int)
test_mae_43 = mean_absolute_error(y_43, y_pred_43)
smape_43 = smape(y_43, y_pred_43)
print(f"2023 Training Metrics")
print(f"mae for location 43: {test_mae_43:.4f};\n smape for location 43:{smape_43}")

y_pred_2024_43 = np.ceil(model.predict(X_2024_43)).astype(int)
test_mae_2024_43 = mean_absolute_error(y_2024_43, y_pred_2024_43)
smape_2024_43 = smape(y_2024_43, y_pred_2024_43)
print(f"2024 Test Metrics")
print(f"mae for location 43: {test_mae_2024_43:.4f};\n smape for location 43:{smape_2024_43}")

#Location 138
model = lgb.LGBMRegressor()
model.fit(X_138, y_138)
y_pred_138 = np.ceil(model.predict(X_138)).astype(int)
test_mae_138 = mean_absolute_error(y_138, y_pred_138)
smape_138 = smape(y_138, y_pred_138)
print(f"2023 Training Metrics")
print(f"mae for location 138: {test_mae_138:.4f};\n smape for location 43:{smape_138}")

y_pred_2024_138 = np.ceil(model.predict(X_2024_138)).astype(int)
test_mae_2024_138 = mean_absolute_error(y_2024_138, y_pred_2024_138)
smape_2024_138 = smape(y_2024_138, y_pred_2024_138)
print(f"2024 Test Metrics")
print(f"mae for location 138: {test_mae_2024_138:.4f};\n smape for location 138:{smape_2024_138}")

#Location 237
model = lgb.LGBMRegressor()
model.fit(X_237, y_237)
y_pred_237 = np.ceil(model.predict(X_237)).astype(int)
test_mae_237 = mean_absolute_error(y_237, y_pred_237)
smape_237 = smape(y_237, y_pred_237)
print(f"2023 Training Metrics")
print(f"mae for location 237: {test_mae_237:.4f};\n smape for location 237:{smape_237}")

y_pred_2024_237 = np.ceil(model.predict(X_2024_237)).astype(int)
test_mae_2024_237 = mean_absolute_error(y_2024_237, y_pred_2024_237)
smape_2024_237 = smape(y_2024_237, y_pred_2024_237)
print(f"2024 Test Metrics")
print(f"mae for location 138: {test_mae_2024_237:.4f};\n smape for location 138:{smape_2024_237}")


In [ ]:
# Create a DataFrame with named columns for better readability
#Location ID 43
df_result_43 = pd.DataFrame({
    'pickup_hour': df_ts_43['pickup_hour'],
    'actual': y_43,
    'predicted': y_pred_43
})
print(df_result_43)

df_result_2024_43 = pd.DataFrame({
    'pickup_hour': df_ts_2024_43['pickup_hour'],
    'actual': y_2024_43,
    'predicted': y_pred_2024_43
})
print(df_result_2024_43)

#Location ID 138
df_result_138 = pd.DataFrame({
    'pickup_hour': df_ts_138['pickup_hour'],
    'actual': y_138,
    'predicted': y_pred_138
})
print(df_result_138)

df_result_2024_138 = pd.DataFrame({
    'pickup_hour': df_ts_2024_138['pickup_hour'],
    'actual': y_2024_138,
    'predicted': y_pred_2024_138
})
print(df_result_2024_138)

#Location ID 237
df_result_237 = pd.DataFrame({
    'pickup_hour': df_ts_237['pickup_hour'],
    'actual': y_237,
    'predicted': y_pred_237
})
print(df_result_237)

df_result_2024_237 = pd.DataFrame({
    'pickup_hour': df_ts_2024_237['pickup_hour'],
    'actual': y_2024_237,
    'predicted': y_pred_2024_237
})
print(df_result_2024_237)

In [ ]:
import plotly.graph_objects as go

def plot_ride_prediction(df_result, final_prediction=None):
    """
    Create a Plotly visualization of actual vs predicted time series with an optional final prediction point.

    Parameters:
    - df_result: DataFrame with columns ['pickup_hour', 'actual', 'predicted']
    - final_prediction: Optional dictionary with keys ['pickup_hour', 'predicted'] for the next prediction
    """
    # Create figure
    fig = go.Figure()

    # Add actual rides trace
    fig.add_trace(go.Scatter(
        x=df_result['pickup_hour'],
        y=df_result['actual'],
        mode='lines',
        name='Actual Rides',
        line=dict(color='blue', width=2)
    ))

    # Add predicted rides trace
    fig.add_trace(go.Scatter(
        x=df_result['pickup_hour'],
        y=df_result['predicted'],
        mode='lines',
        name='Predicted Rides',
        line=dict(color='green', width=2, dash='dash')
    ))

    # Add final prediction point if provided
    if final_prediction:
        fig.add_trace(go.Scatter(
            x=[final_prediction['pickup_hour']],
            y=[final_prediction['predicted']],
            mode='markers',
            name='Next Hour Prediction',
            marker=dict(color='red', size=12, symbol='star')
        ))

        # Add annotation for the prediction
        fig.add_annotation(
            x=final_prediction['pickup_hour'],
            y=final_prediction['predicted'],
            text=f"Prediction: {final_prediction['predicted']} rides",
            showarrow=True,
            arrowhead=1,
            ax=40,
            ay=-40
        )

    # Add vertical line at the end of historical data
    last_historical_time = df_result['pickup_hour'].iloc[-1]
    fig.add_shape(
        type="line",
        x0=last_historical_time,
        y0=0,
        x1=last_historical_time,
        y1=max(df_result['actual'].max(), df_result['predicted'].max()) * 1.1,
        line=dict(color="gray", width=2, dash="dot")
    )

    # Update layout
    fig.update_layout(
        title="Ride Prediction",
        xaxis_title="Date & Time",
        yaxis_title="Number of Rides",
        legend_title="Data Series",
        hovermode="x unified",
        template="plotly_white"
    )

    # Add range selector
    fig.update_xaxes(
        rangeslider_visible=True,
        rangeselector=dict(
            buttons=list([
                dict(count=12, label="12h", step="hour", stepmode="backward"),
                dict(count=1, label="1d", step="day", stepmode="backward"),
                dict(count=7, label="1w", step="day", stepmode="backward"),
                dict(step="all")
            ])
        )
    )

    return fig

In [ ]:
plot_ride_prediction(df_result_43)
plot_ride_prediction(df_result_138)
plot_ride_prediction(df_result_237)

plot_ride_prediction(df_result_2024_43)
plot_ride_prediction(df_result_2024_138)
plot_ride_prediction(df_result_2024_237)


In [ ]:
# Create a DataFrame with named columns for better readability
#Location ID 43
df_predicted_43 = pd.DataFrame({
    'prediction_datetime': df_ts_2024_43['pickup_hour'],
    'predicted_rides': y_pred_2024_43.astype(int)
})
print(df_predicted_43)

#Location ID 138
df_predicted_138 = pd.DataFrame({
    'prediction_datetime': df_ts_2024_138['pickup_hour'],
    'predicted_rides': y_pred_2024_138.astype(int)
})
print(df_predicted_138)

#Location ID 237
df_predicted_237 = pd.DataFrame({
    'prediction_datetime': df_ts_2024_237['pickup_hour'],
    'predicted_rides': y_pred_2024_237.astype(int)
})
print(df_predicted_237)

In [ ]:
import os
import sqlite3
import boto3
import pandas as pd
from io import StringIO
from datetime import datetime

# Function to save predictions to S3 with partitioned structure and SQLite3 database
def save_predictions_to_s3_partitioned(df, location_id, s3_bucket, model):
    """
    Save predictions to S3 or local filesystem using a partitioned folder structure
    and also save the predictions to an SQLite3 database.

    Args:
        df (pandas.DataFrame): DataFrame with predictions
        location_id (int): Pickup location ID
        s3_bucket (str): S3 bucket name.
    """
    # Create S3 client if bucket provided
    s3_client = boto3.client('s3') if s3_bucket else None

    # SQLite3 database path
    sqlite_db_path = "predicted.db"

    # Connect to SQLite3 database (create if it doesn't exist)
    conn = sqlite3.connect(sqlite_db_path)
    cursor = conn.cursor()

    # Create a table for predictions if it doesn't already exist
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        location_id INTEGER,
        prediction_datetime TEXT,
        predicted_rides INTEGER
    )
    """)

    # Process each row and save to appropriate partition
    for _, row in df.iterrows():
        dt = pd.to_datetime(row['prediction_datetime'])
        year = dt.year
        month = dt.month
        day = dt.day
        hour = dt.hour

        # Create partition path
        partition_path = f"model={model}/pickup_location_id={location_id}/year={year}/month={month:02d}/day={day:02d}/hour={hour}"

        # Create a single-row DataFrame for this prediction
        pred_df = pd.DataFrame([row])

        # Convert to CSV
        csv_buffer = StringIO()
        pred_df.to_csv(csv_buffer, index=False)

        if s3_bucket:
            # Upload to S3
            s3_client.put_object(
                Bucket=s3_bucket,
                Key=f"taxi/predictions/{partition_path}/prediction.csv",
                Body=csv_buffer.getvalue()
            )
            print(f"Saved to s3://{s3_bucket}/{partition_path}/prediction.csv")

        # Insert the prediction into the SQLite3 database
        cursor.execute("""
        INSERT INTO predictions (location_id, prediction_datetime, predicted_rides)
        VALUES (?, ?, ?)
        """, (location_id, row['prediction_datetime'], row['predicted_rides']))

    # Commit and close the SQLite3 connection
    conn.commit()
    conn.close()

    print(f"Predictions saved to SQLite3 database at {sqlite_db_path}")

In [ ]:
s3_bucket = "schippa-etl-b06fa59f-6b95-4049-acc2-2ab9390442ba"

save_predictions_to_s3_partitioned(df_predicted_43, 43, s3_bucket, 1)
save_predictions_to_s3_partitioned(df_predicted_138, 138, s3_bucket, 1)
save_predictions_to_s3_partitioned(df_predicted_237, 237, s3_bucket, 1)


## Model 2  LightGBM with feature selection

In [ ]:
df_ts_43_2 = create_time_series_features(filtered_df_43, 24*28)
df_ts_138_2 = create_time_series_features(filtered_df_138, 24*28)
df_ts_237_2 = create_time_series_features(filtered_df_237, 24*28)

df_ts_2024_43_2 = create_time_series_features(filtered_df_2024_43, 24*28)
df_ts_2024_138_2 = create_time_series_features(filtered_df_2024_138, 24*28)
df_ts_2024_237_2 = create_time_series_features(filtered_df_2024_237, 24*28)

In [ ]:
def get_top_features(model, top_n):
    importance = model.feature_importances_
    feature_names = model.feature_name_
    feature_importance = list(zip(feature_names, importance))
    # Sort by importance descending
    sorted_features = sorted(feature_importance, key=lambda x: x[1], reverse=True)
    return [f[0] for f in sorted_features[:top_n]]


In [ ]:
import pandas as pd

def add_avg_hourly_feature(df):
    df = df.copy()
    df['pickup_hour'] = pd.to_datetime(df['pickup_hour'])

    # Extract hour of day and day of week
    df['hour'] = df['pickup_hour'].dt.hour
    df['weekday'] = df['pickup_hour'].dt.weekday  # Monday=0, Sunday=6

    # Create a combined key to group by: (weekday, hour) per location
    df['week_hour'] = df['weekday'].astype(str) + '_' + df['hour'].astype(str)

    # Sort values to ensure correct rolling
    df.sort_values(['location_id', 'pickup_hour'], inplace=True)

    # Group by location and week_hour, then apply rolling mean on target
    df['avg_hourly_last_4_weeks'] = (
        df.groupby(['location_id', 'week_hour'])['target']
        .transform(lambda x: x.shift(1).rolling(window=4, min_periods=1).mean())
    )

    return df


In [ ]:
df_ts_43_2_trans = add_avg_hourly_feature(df_ts_43_2)
df_ts_138_2_trans = add_avg_hourly_feature(df_ts_138_2)
df_ts_237_2_trans = add_avg_hourly_feature(df_ts_237_2)

df_ts_2024_43_2_trans = add_avg_hourly_feature(df_ts_2024_43_2)
df_ts_2024_138_2_trans = add_avg_hourly_feature(df_ts_2024_138_2)
df_ts_2024_237_2_trans = add_avg_hourly_feature(df_ts_2024_237_2)

In [ ]:
# Features and Targets for 2023 model2 training
#For location 43
X_43_2 = df_ts_43_2_trans.copy()
X_43_2 = X_43_2.drop(columns=['pickup_hour', 'target'])
y_43_2 = df_ts_43_2_trans['target']

#For location 138
X_138_2 = df_ts_138_2_trans.copy()
X_138_2 = X_138_2.drop(columns=['pickup_hour', 'target'])
y_138_2 = df_ts_138_2_trans['target']

#For location 237
X_237_2 = df_ts_138_2_trans.copy()
X_237_2 = X_237_2.drop(columns=['pickup_hour', 'target'])
y_237_2 = df_ts_138_2_trans['target']


#Features and Targets for 2024 Predictions
#For location 43
X_2024_43_2 = df_ts_2024_43_2_trans.copy()
X_2024_43_2 = X_2024_43_2.drop(columns=['pickup_hour', 'target'])
y_2024_43_2 = df_ts_2024_43_2_trans['target']

#For location 138
X_2024_138_2 = df_ts_2024_138_2_trans.copy()
X_2024_138_2 = X_2024_138_2.drop(columns=['pickup_hour', 'target'])
y_2024_138_2 = df_ts_2024_138_2_trans['target']

#For location 237
X_2024_237_2 = df_ts_2024_237_2_trans.copy()
X_2024_237_2 = X_2024_237_2.drop(columns=['pickup_hour', 'target'])
y_2024_237_2 = df_ts_2024_237_2_trans['target']


In [ ]:
required_features = ['lag_1', 'lag_24', 'lag_48', 'lag_72', 'lag_96', 'avg_hourly_last_4_weeks']
#Location ID 43
model = lgb.LGBMRegressor()
model.fit(X_43_2, y_43_2)

top5_43 = get_top_features(model, 5)
final_features_43 = required_features + top_5_43


#Location ID 138
model = lgb.LGBMRegressor()
model.fit(X_138_2, y_138_2)

top5_138 = get_top_features(model, 5)
final_features = required_features + top_5_138


#Location ID 237
model = lgb.LGBMRegressor()
model.fit(X_237_2, y_237_2)

top5_237 = get_top_features(model, 5)
final_features = required_features + top_5_237


In [ ]:
def train_model(df_location, features):
    X = df_location[features]
    y = df_location['target']
    model = lgb.LGBMRegressor()
    model.fit(X, y)
    return model


In [ ]:
model2_43 = train_model_2(df_ts_43_2_trans, final_features_43)
model2_138 = train_model_2(df_ts_138_2_trans, final_features_138)
model2_237 = train_model_2(df_ts_237_2_trans, final_features_237)


In [ ]:
#Location 43
y_pred_43 = np.ceil(model2_43.predict(X_43_2)).astype(int)
test_mae_43_2 = mean_absolute_error(y_43_2, y_pred_43_2)
smape_43_2 = smape(y_43_2, y_pred_43_2)
print(f"2023 Training Metrics")
print(f"mae for location 43: {test_mae_43_2:.4f};\n smape for location 43:{smape_43_2}")

y_pred_2024_43_2 = np.ceil(model2_43.predict(X_2024_43_2)).astype(int)
test_mae_2024_43_2 = mean_absolute_error(y_2024_43_2, y_pred_2024_43_2)
smape_2024_43_2 = smape(y_2024_43_2, y_pred_2024_43_2)
print(f"2024 Test Metrics")
print(f"mae for location 43: {test_mae_2024_43_2:.4f};\n smape for location 43:{smape_2024_43_2}")

#Location 138
y_pred_138_2 = np.ceil(model2_138.predict(X_138_2)).astype(int)
test_mae_138_2 = mean_absolute_error(y_138_2, y_pred_138_2)
smape_138_2 = smape(y_138_2, y_pred_138_2)
print(f"2023 Training Metrics")
print(f"mae for location 138: {test_mae_138_2:.4f};\n smape for location 43:{smape_138_2}")

y_pred_2024_138_2 = np.ceil(model2_138.predict(X_2024_138_2)).astype(int)
test_mae_2024_138_2 = mean_absolute_error(y_2024_138_2, y_pred_2024_138_2)
smape_2024_138_2 = smape(y_2024_138_2, y_pred_2024_138_2)
print(f"2024 Test Metrics")
print(f"mae for location 138: {test_mae_2024_138_2:.4f};\n smape for location 138:{smape_2024_138_2}")

#Location 237
y_pred_237_2 = np.ceil(model2_237.predict(X_237_2)).astype(int)
test_mae_237_2 = mean_absolute_error(y_237_2, y_pred_237_2)
smape_237_2 = smape(y_237_2, y_pred_237_2)
print(f"2023 Training Metrics")
print(f"mae for location 237: {test_mae_237_2:.4f};\n smape for location 237:{smape_237_2}")

y_pred_2024_237_2 = np.ceil(model2_237.predict(X_2024_237_2)).astype(int)
test_mae_2024_237_2 = mean_absolute_error(y_2024_237_2, y_pred_2024_237_2)
smape_2024_237_2 = smape(y_2024_237_2, y_pred_2024_237_2)
print(f"2024 Test Metrics")
print(f"mae for location 138: {test_mae_2024_237_2:.4f};\n smape for location 138:{smape_2024_237_2}")

In [ ]:
#Prediction DataFrames for Model2

#Location ID 43
df_predicted_43_2 = pd.DataFrame({
    'prediction_datetime': df_ts_2024_43_2_trans['pickup_hour'],
    'predicted_rides': y_pred_2024_43_2.astype(int)
})
print(df_predicted_43_2)

#Location ID 138
df_predicted_138_2 = pd.DataFrame({
    'prediction_datetime': df_ts_2024_138_2_trans['pickup_hour'],
    'predicted_rides': y_pred_2024_138_2.astype(int)
})
print(df_predicted_138_2)

#Location ID 237
df_predicted_237_2 = pd.DataFrame({
    'prediction_datetime': df_ts_2024_237_2_trans['pickup_hour'],
    'predicted_rides': y_pred_2024_237_2.astype(int)
})
print(df_predicted_237_2)

In [ ]:
s3_bucket = "schippa-etl-b06fa59f-6b95-4049-acc2-2ab9390442ba"

save_predictions_to_s3_partitioned(df_predicted_43_2, 43, s3_bucket, 2)
save_predictions_to_s3_partitioned(df_predicted_138_2, 138, s3_bucket, 2)
save_predictions_to_s3_partitioned(df_predicted_237_2, 237, s3_bucket, 2)